In [ ]:
load_date = "2014-05-02"

In [ ]:
from pyspark.sql.functions import current_timestamp, lit

input_path = (
    "Files/landing/"
    f"SalesOrderHeader/load_date={load_date}/"
    "SalesOrderHeader.parquet"
)

bronze_sales_order_header = (
    spark.read.parquet(input_path)
    .withColumn("load_date", lit(load_date).cast("date"))
    .withColumn("source_system", lit("adls_landing"))
    .withColumn("ingestion_timestamp", current_timestamp())
)

print(f"Lignes du jour : {bronze_sales_order_header.count()}")
print(f"Date utilisée : {load_date}")

(
    bronze_sales_order_header.write
    .mode("overwrite")
    .option("replaceWhere", f"load_date = '{load_date}'")
    .format("delta")
    .saveAsTable("bronze_salesorderheader")
)

bronze_check = spark.read.table("bronze_salesorderheader")

print(f"Total Bronze après traitement : {bronze_check.count()}")
print("Partition quotidienne remplacée avec succès.")

In [ ]:
display(
    bronze_sales_order_header
    .groupBy("load_date")
    .count()
    .orderBy("load_date")
)